# 19A4E3 — 2026 AIA Production / Input Verification

## Purpose

Verify the currently available **2026 AIA production-v1 data** before any 2026
supplementary performance claim is made.

This notebook checks:

1. **Production contract** — expected six-channel order, shape, dtype, finite values and source lineage.
2. **Input distribution** — whether 2026 resembles 2025 production-v1 data or shows another shift.

This is diagnostic only. It does not retrain the model, refit calibration,
reselect the threshold, modify the frozen 2021–2025 result, or merge 2026 into
the primary test.


In [1]:
from pathlib import Path
import json, re, subprocess, tempfile
from collections import Counter
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp

HOME = Path.home()
OUT = HOME / "aia19_2026_input_verification"
OUT.mkdir(parents=True, exist_ok=True)

PREP = HOME / "aia19_cycle25_staging_prep"
NORM_PATH = HOME / "aia19_cnn_gru_20260917" / "normalisation.json"

GCS_PROD_ROOT = (
    "gs://suryabench-sharp-pipeline-bamidele/"
    "jsoc_2025_2026_production_v1/samples_npz"
)

EXPECTED_WAVELENGTHS = [94,131,171,193,211,335]
EXPECTED_CHANNELS = ["aia94","aia131","aia171","aia193","aia211","aia335"]

SAMPLE_OBJECTS_PER_YEAR = 400
PIXELS_PER_OBJECT = 2048
SEED = 20260919
rng = np.random.default_rng(SEED)

assert NORM_PATH.exists(), NORM_PATH
norm = json.loads(NORM_PATH.read_text())
scale = np.asarray(norm["channel_scale"], dtype=np.float32)
mean = np.asarray(norm["channel_mean"], dtype=np.float32)
std = np.asarray(norm["channel_std"], dtype=np.float32)

print("Output:", OUT)


Output: /home/abmoses2000/aia19_2026_input_verification


## 1. Discover 2025 and 2026 object URIs

In [2]:
def read_uri_columns(path):
    try:
        d = pd.read_csv(path, low_memory=False)
    except Exception:
        return []
    uris = []
    for col in d.columns:
        s = d[col].astype(str)
        mask = s.str.startswith("gs://", na=False) & s.str.contains("samples_npz", regex=False, na=False)
        if mask.any():
            uris.extend(s[mask].tolist())
    return uris

inv25 = PREP / "cycle25_2025_unique_aia_objects.csv.gz"
assert inv25.exists(), inv25
d25 = pd.read_csv(inv25)
assert "object_uri" in d25.columns
uris25 = sorted(d25["object_uri"].dropna().astype(str).unique().tolist())

uris26 = []
sources = []

candidate_paths = [
    PREP / "cycle25_2026_unique_aia_objects.csv.gz",
    HOME / "aia19_temporal_aia_input_corrected" / "cycle25_2026_unique_aia_objects.csv.gz",
    HOME / "aia19_temporal_aia_input_corrected" / "supplementary_2026_unique_aia_objects.csv.gz",
]

for p in candidate_paths:
    if p.exists():
        d = pd.read_csv(p)
        u = d["object_uri"].dropna().astype(str).tolist() if "object_uri" in d.columns else read_uri_columns(p)
        if u:
            uris26.extend(u)
            sources.append(str(p))

if not uris26:
    for root in [HOME / "aia19_temporal_aia_input_corrected", HOME / "aia19_cycle25_staging_prep"]:
        if not root.exists():
            continue
        for p in root.rglob("*2026*.csv*"):
            if p.stat().st_size > 300_000_000:
                continue
            u = read_uri_columns(p)
            if u:
                uris26.extend(u)
                sources.append(str(p))

if not uris26:
    print("No local 2026 manifest found; listing production-v1 GCS objects...")
    proc = subprocess.run(
        ["gcloud","storage","ls","--recursive",GCS_PROD_ROOT],
        check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
    )
    all_prod = [x.strip() for x in proc.stdout.splitlines() if x.strip().endswith(".npz")]
    for u in all_prod:
        b = u.rsplit("/",1)[-1]
        if "2026-" in b or "2026_" in b or "2026." in b or "/2026/" in u:
            uris26.append(u)
    sources.append("recursive GCS listing filtered by visible 2026 token")

uris26 = sorted(set(uris26))

print("2025 URIs:", len(uris25))
print("2026 candidate URIs:", len(uris26))
print("Sources:")
for s in sources[:20]:
    print(" -", s)

if not uris26:
    raise RuntimeError("Could not discover 2026 object URIs automatically.")


2025 URIs: 10879
2026 candidate URIs: 2490
Sources:
 - /home/abmoses2000/aia19_temporal_aia_input_corrected/aia_2026_supplementary_manifest.csv.gz


In [3]:
def local_path_for_uri(year, uri):
    fn = uri.rsplit("/",1)[-1]
    candidates = [
        Path("/mnt/disks/aia-cache/cycle25_yearwise") / str(year) / fn,
        Path("/mnt/disks/aia-cache/cycle25") / str(year) / fn,
        Path("/mnt/disks/aia-cache/2026") / fn,
    ]
    for p in candidates:
        if p.exists():
            return p
    return None

def fetch(uri, td):
    dest = Path(td) / uri.rsplit("/",1)[-1]
    subprocess.run(
        ["gcloud","storage","cp","--quiet",uri,str(dest)],
        check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
    )
    return dest

def get_year_trec(path):
    with np.load(path, allow_pickle=False) as z:
        if "T_REC_dt" not in z.files:
            return None, None
        v = z["T_REC_dt"]
        v = v.item() if np.ndim(v) == 0 else v.tolist()
        txt = str(v)
        m = re.search(r"(20\d{2})", txt)
        return (int(m.group(1)) if m else None), txt

verified26 = []
with tempfile.TemporaryDirectory(prefix="verify_2026_") as td:
    for idx in rng.permutation(len(uris26)):
        uri = uris26[int(idx)]
        local = local_path_for_uri(2026, uri)
        fetched = False
        if local is None:
            local = fetch(uri, td)
            fetched = True
        yr, trec = get_year_trec(local)
        if yr == 2026:
            verified26.append((uri, trec))
        if fetched:
            local.unlink(missing_ok=True)
        if len(verified26) >= min(SAMPLE_OBJECTS_PER_YEAR, len(uris26)):
            break

assert verified26, "No internally verified 2026 objects found."

v26 = pd.DataFrame(verified26, columns=["object_uri","T_REC_dt"])
v26.to_csv(OUT / "verified_2026_object_sample.csv", index=False)

print("Internally verified 2026:", len(v26))
print("Sample time range:", v26["T_REC_dt"].min(), "→", v26["T_REC_dt"].max())


Internally verified 2026: 400
Sample time range: 2026-01-02 09:48:00 → 2026-04-03 21:36:00


## 2. Deterministic 2025 reference sample

In [4]:
n25 = min(SAMPLE_OBJECTS_PER_YEAR, len(uris25))
sel25 = rng.choice(len(uris25), size=n25, replace=False)
sample25 = [uris25[i] for i in np.sort(sel25)]
sample26 = v26["object_uri"].tolist()[:SAMPLE_OBJECTS_PER_YEAR]

print("2025 sample:", len(sample25))
print("2026 sample:", len(sample26))


2025 sample: 400
2026 sample: 400


## 3. Production/schema verification and pixel statistics

In [5]:
def metadata_contract(z):
    if "wavelengths" in z.files:
        return "wavelengths", [int(v) for v in z["wavelengths"].tolist()]
    if "channels" in z.files:
        return "channels", [str(v) for v in z["channels"].tolist()]
    return "none", None

def scalar_text(z, key):
    if key not in z.files:
        return None
    v = z[key]
    try:
        if np.ndim(v) == 0:
            return str(v.item())
        return json.dumps(v.tolist(), default=str)
    except Exception:
        return repr(v)

object_rows = []
pixel_samples = {2025:[[] for _ in range(6)], 2026:[[] for _ in range(6)]}
tx_samples = {2025:[[] for _ in range(6)], 2026:[[] for _ in range(6)]}
schema_counts = {2025:Counter(), 2026:Counter()}

for year, sample in [(2025,sample25),(2026,sample26)]:
    with tempfile.TemporaryDirectory(prefix=f"aia_{year}_verify_") as td:
        print(f"\n===== {year} =====", flush=True)
        for i, uri in enumerate(sample):
            local = local_path_for_uri(year, uri)
            fetched = False
            if local is None:
                local = fetch(uri, td)
                fetched = True

            with np.load(local, allow_pickle=False) as z:
                schema = tuple(sorted(z.files))
                schema_counts[year][schema] += 1

                assert "x" in z.files
                x = z["x"]
                assert x.shape == (512,512,6), (uri, x.shape)
                assert x.dtype == np.float32, (uri, x.dtype)
                assert np.isfinite(x).all(), uri

                mode, val = metadata_contract(z)
                if mode == "wavelengths":
                    assert val == EXPECTED_WAVELENGTHS, (uri, val)
                elif mode == "channels":
                    assert val == EXPECTED_CHANNELS, (uri, val)
                else:
                    raise RuntimeError(f"No channel metadata: {uri}")

                yr, trec = get_year_trec(local)
                assert yr == year, (uri, trec)

                flat = x.reshape(-1,6)
                take = rng.choice(flat.shape[0], size=min(PIXELS_PER_OBJECT, flat.shape[0]), replace=False)
                pix = flat[take].astype(np.float64, copy=False)

                tx = np.arcsinh(pix / scale.reshape(1,6))
                tx = (tx - mean.reshape(1,6)) / std.reshape(1,6)

                rec = {
                    "year": year,
                    "uri": uri,
                    "T_REC_dt": trec,
                    "metadata_mode": mode,
                    "schema": "|".join(schema),
                    "source": scalar_text(z, "source"),
                    "raw_global_mean": float(np.mean(x)),
                    "raw_global_std": float(np.std(x)),
                    "raw_zero_fraction": float(np.mean(x == 0)),
                }

                for c in range(6):
                    pixel_samples[year][c].append(pix[:,c])
                    tx_samples[year][c].append(tx[:,c])
                    rec[f"ch{c}_raw_median"] = float(np.median(pix[:,c]))
                    rec[f"ch{c}_tx_median"] = float(np.median(tx[:,c]))

                object_rows.append(rec)

            if fetched:
                local.unlink(missing_ok=True)

            if (i+1) % 50 == 0 or (i+1) == len(sample):
                print(f"{year}: {i+1}/{len(sample)}", flush=True)

objects = pd.DataFrame(object_rows)
objects.to_csv(OUT / "2025_2026_object_level_input_statistics.csv.gz", index=False, compression="gzip")

print("\nSCHEMA COUNTS")
for year in [2025,2026]:
    print("\nYEAR", year)
    for schema, n in schema_counts[year].most_common():
        print(n, schema)



===== 2025 =====


2025: 50/400


2025: 100/400


2025: 150/400


2025: 200/400


...

2026: 250/400


2026: 300/400


2026: 350/400


2026: 400/400



SCHEMA COUNTS

YEAR 2025
400 ('HARPNUM', 'NOAA_AR_clean', 'T_REC_dt', 'block_id', 'channel_metadata', 'label_48h_final', 'sample_id', 'source', 'wavelengths', 'x', 'y')

YEAR 2026
400 ('HARPNUM', 'NOAA_AR_clean', 'T_REC_dt', 'block_id', 'channel_metadata', 'label_48h_final', 'sample_id', 'source', 'wavelengths', 'x', 'y')


## 4. Channel-level 2025 vs 2026 comparison

In [6]:
summary_rows = []
ks_rows = []

for c, ch in enumerate(EXPECTED_CHANNELS):
    a_raw = np.concatenate(pixel_samples[2025][c])
    b_raw = np.concatenate(pixel_samples[2026][c])
    a_tx = np.concatenate(tx_samples[2025][c])
    b_tx = np.concatenate(tx_samples[2026][c])

    for year, raw, tx in [(2025,a_raw,a_tx),(2026,b_raw,b_tx)]:
        summary_rows.append({
            "year": year,
            "channel_index": c,
            "channel": ch,
            "n_pixels": int(len(raw)),
            "raw_mean": float(np.mean(raw)),
            "raw_std": float(np.std(raw)),
            "raw_q01": float(np.quantile(raw,.01)),
            "raw_q50": float(np.quantile(raw,.50)),
            "raw_q99": float(np.quantile(raw,.99)),
            "raw_zero_fraction": float(np.mean(raw == 0)),
            "transformed_mean": float(np.mean(tx)),
            "transformed_std": float(np.std(tx)),
            "transformed_q01": float(np.quantile(tx,.01)),
            "transformed_q50": float(np.quantile(tx,.50)),
            "transformed_q99": float(np.quantile(tx,.99)),
        })

    for variable, aa, bb in [("raw",a_raw,b_raw),("frozen_preprocessed",a_tx,b_tx)]:
        stat, p = ks_2samp(aa,bb)
        ks_rows.append({
            "channel_index": c,
            "channel": ch,
            "variable": variable,
            "ks_statistic": float(stat),
            "p_value": float(p),
            "n_2025": int(len(aa)),
            "n_2026": int(len(bb)),
        })

summary = pd.DataFrame(summary_rows)
ks = pd.DataFrame(ks_rows)

summary.to_csv(OUT / "channel_distribution_summary_2025_vs_2026.csv", index=False)
ks.to_csv(OUT / "channel_ks_2025_vs_2026.csv", index=False)

print(summary.to_string(index=False))
print("\nKS TESTS")
print(ks.to_string(index=False))


 year  channel_index channel  n_pixels  raw_mean  raw_std  raw_q01  raw_q50  raw_q99  raw_zero_fraction  transformed_mean  transformed_std  transformed_q01  transformed_q50  transformed_q99
 2025              0   aia94    819200  0.412999 0.141265 0.119022 0.404594 0.778596           0.003140          1.002879         0.890120        -1.194109         1.035402         2.915645
 2026              0   aia94    819200  0.418138 0.139232 0.127804 0.410379 0.782092           0.002694          1.038194         0.871547        -1.113307         1.072091         2.929553
 2025              1  aia131    819200  0.419163 0.139739 0.126282 0.406114 0.787362           0.003896          0.534006         0.993054        -1.873467         0.518115         2.780514
 2026              1  aia131    819200  0.405231 0.143378 0.074881 0.391509 0.785606           0.006030          0.427864         1.036189        -2.378579         0.410405         2.772065
 2025              2  aia171    819200  0.377666 0

## 5. Production contract summary

In [7]:
prod = (
    objects.groupby(["year","metadata_mode","schema","source"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(["year","count"], ascending=[True,False])
)

prod.to_csv(OUT / "production_contract_summary_2025_vs_2026.csv", index=False)
print(prod.to_string(index=False))


 year metadata_mode                                                                                                    schema                                            source  count
 2025   wavelengths HARPNUM|NOAA_AR_clean|T_REC_dt|block_id|channel_metadata|label_48h_final|sample_id|source|wavelengths|x|y JSOC HARP-block tracked im_patch + local WCS crop    400
 2026   wavelengths HARPNUM|NOAA_AR_clean|T_REC_dt|block_id|channel_metadata|label_48h_final|sample_id|source|wavelengths|x|y JSOC HARP-block tracked im_patch + local WCS crop    400


## 6. Verification record

In [8]:
record = {
    "status": "2026_AIA_PRODUCTION_INPUT_VERIFICATION_COMPLETE_NO_TUNING",
    "primary_2021_2025_result_unchanged": True,
    "sample_objects_2025": int(len(sample25)),
    "sample_objects_2026": int(len(sample26)),
    "pixels_per_object": PIXELS_PER_OBJECT,
    "seed": SEED,
    "expected_wavelengths": EXPECTED_WAVELENGTHS,
    "model_weights_changed": False,
    "normalisation_changed": False,
    "calibrator_changed": False,
    "threshold_changed": False,
    "2026_merged_into_primary_test": False,
    "notes": [
        "2026 remains supplementary.",
        "This verifies production/schema/input-distribution behavior only.",
        "Final 2026 frozen performance must be evaluated separately after the September refresh is complete.",
        "Distribution differences do not by themselves identify physical cause."
    ],
}

(OUT / "verification_record.json").write_text(json.dumps(record, indent=2) + "\n")
print(json.dumps(record, indent=2))
print("\n19A4E3_2026_INPUT_VERIFICATION_COMPLETE")


{
  "status": "2026_AIA_PRODUCTION_INPUT_VERIFICATION_COMPLETE_NO_TUNING",
  "primary_2021_2025_result_unchanged": true,
  "sample_objects_2025": 400,
  "sample_objects_2026": 400,
  "pixels_per_object": 2048,
  "seed": 20260919,
  "expected_wavelengths": [
    94,
    131,
    171,
    193,
    211,
    335
  ],
  "model_weights_changed": false,
  "normalisation_changed": false,
  "calibrator_changed": false,
  "threshold_changed": false,
  "2026_merged_into_primary_test": false,
  "notes": [
    "2026 remains supplementary.",
    "This verifies production/schema/input-distribution behavior only.",
    "Final 2026 frozen performance must be evaluated separately after the September refresh is complete.",
    "Distribution differences do not by themselves identify physical cause."
  ]
}

19A4E3_2026_INPUT_VERIFICATION_COMPLETE
